# Vignette 01: Quickstart

**mlsurv** is a low-code survival machine learning package. This vignette walks through a minimal end-to-end workflow: load data, compare models, inspect results, and visualize then generate an HTML report.

*Runtime: ~2 minutes on a modern laptop.*

In [ ]:
import os

from mlsurv import SurvivalLearner
from mlsurv.utils import load_survival_data

# Results are written to ./output/
os.makedirs("output", exist_ok=True)

## Load the METABRIC dataset

METABRIC (Curtis et al. 2012, as preprocessed in Katzman et al. 2018) is a breast cancer dataset with 1,903 patients, 9 clinical/gene-expression features, and an event indicator for overall survival.

In [ ]:
import pandas as pd
X, y = load_survival_data('data/metabric.csv', time_col='duration', event_col='event')
print(f"METABRIC: n={len(X)}, features={X.shape[1]}, events={int(y['event'].sum())}")

alldata_df = X.join(pd.DataFrame(y, index=X.index))
alldata_df.head(5)

## Configure the pipeline

`setup()` prepares the train/test split, imputation, scaling, and cross-validation strategy. It also runs proportional-hazards and log-hazard linearity diagnostics on the training data and consolidates them into a single **model recommendations table**. The table classifies every model into *Recommended / Cautionary / Not recommended* tiers based on:

- Dataset size and events-per-variable (EPV)
- PH assumption violations (Cox-family models are demoted when features violate PH)
- Log-linearity violations (Cox-family models are demoted when continuous features are non-linear in the log-hazard)
- A high-dimensional rule that disqualifies unregularized Cox models when `n_features >= n_events`

Defaults are sensible for most datasets — here we only set the test fraction and random seed.

In [ ]:
learner = SurvivalLearner(X, y)
learner.setup(test_fraction=0.25, random_state=42, n_jobs=1)

## Compare models

`run()` tunes hyperparameters where applicable (via [Optuna](https://optuna.org); Akiba et al. 2019), runs cross-validation, evaluates on the held-out test set, and computes bootstrap confidence intervals — all in one call. We compare CoxPH (classical semi-parametric; Cox 1972) with XGBoost-AFT (gradient-boosted trees; Chen & Guestrin 2016, Barnwal et al. 2022).

`run()` returns a dict mapping each model name to its `ModelResult`. That dict holds the same `ModelResult` objects as `learner.models`, so you can fetch a model at any time with `learner.models['coxph']` (or the winner with `learner.best_model`) without keeping the return value. 

In [ ]:
learner.run(models=['coxph', 'xgbaft'])

## Visualize risk stratification

Kaplan-Meier curves (Kaplan & Meier 1958) for predicted risk groups show whether models separate patients into meaningfully different survival trajectories. With no arguments, `plot_survival()` renders a panel grid — one subplot per model.

In [ ]:
learner.plot_survival()

## Feature importance

Permutation importance (Breiman 2001) shows which features most influence each model's predictions. With no arguments, `plot_importance()` renders a grouped bar chart comparing all models side by side.

In [ ]:
learner.plot_importance()

## Model performance

Use `plot_metric()` to compare model performance. By default this shows Harrell's C-index (Harrell et al. 1982), a rank-based measure of how well predicted risk orderings agree with observed event times (0.5 = chance, 1.0 = perfect concordance).

In [ ]:
learner.plot_metric()

## Generate report

A self-contained HTML report aggregates all tables, metrics, and visualizations into a single file for sharing.

In [ ]:
learner.generate_report('output/vig1_report.html', title='Quickstart Report');

## Save and export

`save_learner()` serializes the entire session — data, models, evaluation results, bootstrap CIs — so you can resume later without re-running anything. `export_results()` writes all result tables to a single Excel workbook for sharing or downstream analysis.

In [ ]:
learner.save_learner('output/vig1.learner')
learner.export_results('output/vig1_results.xlsx')

After reloading, your per-model results are reachable through the same read-only accessors, so nothing has to be recomputed:

```python
from mlsurv import load_learner

learner = load_learner('output/vig1.learner')
learner.models['coxph']       # per-model ModelResult (survives the round-trip)
learner.best_model.name       # winning model from CV
learner.evaluation_result     # combined EvaluationResult
learner.bootstrap_result      # combined BootstrapResult
```

## Next steps

This quickstart covered the minimal workflow. **Vignette 02 (Workflow Tour)** goes deeper: per-model tuning, SHAP explanations, external validation, and additional plot types.

## References

- Akiba T, Sano S, Yanase T, Ohta T, Koyama M (2019). Optuna: A next-generation hyperparameter optimization framework. *Proc. 25th ACM SIGKDD Int. Conf. on Knowledge Discovery & Data Mining*, 2623–2631. https://doi.org/10.1145/3292500.3330701
- Barnwal A, Cho H, Hocking T (2022). Survival regression with accelerated failure time model in XGBoost. *Journal of Computational and Graphical Statistics*, 31(4):1292–1302. https://doi.org/10.1080/10618600.2022.2067548
- Breiman L (2001). Random forests. *Machine Learning*, 45(1):5–32. https://doi.org/10.1023/A:1010933404324
- Chen T, Guestrin C (2016). XGBoost: A scalable tree boosting system. *Proc. 22nd ACM SIGKDD Int. Conf. on Knowledge Discovery and Data Mining*, 785–794. https://doi.org/10.1145/2939672.2939785
- Cox DR (1972). Regression models and life-tables. *Journal of the Royal Statistical Society, Series B*, 34(2):187–220. https://doi.org/10.1111/j.2517-6161.1972.tb00899.x
- Curtis C, Shah SP, Chin SF, et al. (2012). The genomic and transcriptomic architecture of 2,000 breast tumours reveals novel subgroups. *Nature*, 486(7403):346–352. https://doi.org/10.1038/nature10983
- Harrell FE, Califf RM, Pryor DB, Lee KL, Rosati RA (1982). Evaluating the yield of medical tests. *JAMA*, 247(18):2543–2546. https://doi.org/10.1001/jama.1982.03320430047030
- Kaplan EL, Meier P (1958). Nonparametric estimation from incomplete observations. *Journal of the American Statistical Association*, 53(282):457–481. https://doi.org/10.1080/01621459.1958.10501452
- Katzman JL, Shaham U, Cloninger A, Bates J, Jiang T, Kluger Y (2018). DeepSurv: personalized treatment recommender system using a Cox proportional hazards deep neural network. *BMC Medical Research Methodology*, 18:24. https://doi.org/10.1186/s12874-018-0482-1